══════════════════════════════════════════════════════════════
 WEEK 8  |  DAY 5  |  FINAL PROJECT: END-TO-END DATA ENGINEERING PIPELINE

══════════════════════════════════════════════════════════════

 LEARNING OBJECTIVES
 ───────────────────
 This project integrates everything from Weeks 2-8:
 extract -> transform -> validate -> load -> query -> quality check -> report

 TIME:  ~30 minutes

 YOUTUBE
 ───────
 Search: "Python end to end data pipeline project"
 Search: "ETL pipeline sqlite pandas complete project"

 ─────────────────────────────────────────────────────────────
 ARCHITECTURE DECISION
 ─────────────────────
 Choosing between: FastAPI vs Flask vs Django REST Framework.
 Rule of thumb: use FastAPI for new APIs — async, automatic docs, type-safe. Move to Django REST only when you need its full ORM and admin panel ecosystem.

══════════════════════════════════════════════════════════════

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import csv
import os
import io
import logging
from datetime import datetime

this_dir = os.path.dirname(__file__)

# Set up pipeline logger
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("final_project")

print("=" * 70)
print("FINAL PROJECT: END-TO-END DATA ENGINEERING PIPELINE")
print("=" * 70)

══════════════════════════════════════════════════════════════
 STEP 1 — GENERATE SOURCE CSV FILES

══════════════════════════════════════════════════════════════
In production these would already exist on disk or S3.
We generate them programmatically for a self-contained project.

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
np.random.seed(42)

def generate_region_csv(region, num_rows, seed_offset):
    """Generate a sales CSV string for one region."""
    np.random.seed(42 + seed_offset)
    products = ["Laptop", "Monitor", "Keyboard", "Mouse", "Headset", "Webcam"]
    reps = {
        "West":    ["Alice Ng",   "Lena Kim",   "Beth Morris"],
        "East":    ["Bob Chen",   "Priya Mehta", "Omar Nasser"],
        "Central": ["Carol Diaz", "Sara Jones",  "Mike Brown"],
    }
    region_reps = reps.get(region, ["Rep A", "Rep B"])

    rows = []
    for i in range(num_rows):
        qty   = np.random.randint(1, 10)
        price = round(float(np.random.choice([899.99, 349.99, 89.99, 39.99, 149.99, 69.99])), 2)
        # Introduce some intentional quality issues
        if i % 20 == 0:
            qty = -1          # invalid: negative quantity
        if i % 30 == 0:
            price = None      # missing price
        rows.append({
            "sale_id":   (seed_offset * 1000) + i + 1,
            "sale_date": (pd.Timestamp("2024-01-01") + pd.Timedelta(days=i % 90)).strftime("%Y-%m-%d"),
            "region":    region,
            "rep_name":  np.random.choice(region_reps),
            "product":   np.random.choice(products),
            "quantity":  qty,
            "unit_price": price,
        })
    df = pd.DataFrame(rows)
    return df.to_csv(index=False)

# Create three regional CSV files
source_files = {}
for region, offset, count in [("West", 1, 150), ("East", 2, 180), ("Central", 3, 120)]:
    filename = f"sales_{region.lower()}_2024.csv"
    filepath = os.path.join(this_dir, filename)
    csv_content = generate_region_csv(region, count, offset)
    with open(filepath, "w") as f:
        f.write(csv_content)
    source_files[region] = filepath
    logger.info(f"Generated: {filename} ({count} rows)")

print(f"\nGenerated {len(source_files)} source files.")

══════════════════════════════════════════════════════════════
 TASK 1 — EXTRACT: READ ALL THREE CSV FILES AND COMBINE THEM

══════════════════════════════════════════════════════════════
Instructions:
  1. Read each file in source_files.values() using pd.read_csv()
  2. Add a "_source_file" column to each (os.path.basename of the path)
  3. Combine with pd.concat(ignore_index=True)
  4. Store as df_raw
  5. Print: "Extracted <n> rows from <k> files"

Expected output:
  Extracted 450 rows from 3 files

In [ ]:
logger.info("TASK 1: Extract")
print("\n" + "="*70)
print("TASK 1: EXTRACT")
print("="*70)

══════════════════════════════════════════════════════════════
 STEP 2 — INSPECT RAW DATA (demonstrated for you)

══════════════════════════════════════════════════════════════
This block runs automatically to show you what the raw data looks like.

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
try:
    print("\nRaw data preview:")
    print(df_raw.head(5))
    print(f"\nShape: {df_raw.shape}")
    print(f"Nulls per column:\n{df_raw.isnull().sum()}")
    print(f"\nQuantity stats:\n{df_raw['quantity'].describe()}")
except NameError:
    print("Note: df_raw not yet defined. Complete Task 1 first.")
    # Create a fallback so later steps have data to work with
    df_raw = pd.read_csv(list(source_files.values())[0])
    for path in list(source_files.values())[1:]:
        df_raw = pd.concat([df_raw, pd.read_csv(path)], ignore_index=True)

══════════════════════════════════════════════════════════════
 TASK 2 — TRANSFORM: CLEAN AND ENRICH THE DATA

══════════════════════════════════════════════════════════════
Instructions:
  1. Start with df_raw.copy() -> df_clean
  2. Convert sale_date to datetime (pd.to_datetime, errors="coerce")
  3. Convert quantity and unit_price to numeric (pd.to_numeric, errors="coerce")
  4. Drop rows where quantity <= 0 or unit_price is null/NaN
  5. Calculate "revenue" = quantity * unit_price (round to 2)
  6. Add "quarter" column: "Q1" for months 1-3, "Q2" for 4-6, etc.
     (use df_clean["sale_date"].dt.quarter.map({1:"Q1",2:"Q2",3:"Q3",4:"Q4"}))
  7. Add "deal_size" column:
     revenue < 200 -> "Small", 200-999 -> "Medium", 1000+ -> "Large"
     (use pd.cut with bins=[0, 200, 999, float("inf")], labels=["Small","Medium","Large"])
  8. Standardize rep_name: .str.strip().str.title()
  9. Print: "Transformed: <n> rows remain after cleaning"

Expected output:
  Transformed: ~420 rows remain after cleaning

In [ ]:
logger.info("TASK 2: Transform")
print("\n" + "="*70)
print("TASK 2: TRANSFORM")
print("="*70)

══════════════════════════════════════════════════════════════
 TASK 3 — LOAD: WRITE TO SQLite DATABASE

══════════════════════════════════════════════════════════════
Instructions:
  1. Create an in-memory SQLite database: conn = sqlite3.connect(":memory:")
  2. Write df_clean to a table called "fact_sales" using to_sql()
     with if_exists="replace" and index=False
  3. Create a summary view by running SQL and storing in df_summary:
     SELECT region, quarter, COUNT(*) as sales_count,
            SUM(revenue) as total_revenue,
            AVG(revenue) as avg_revenue
     FROM fact_sales
     GROUP BY region, quarter
     ORDER BY region, quarter
  4. Print the df_summary table

Expected output:
  Loaded <n> rows to fact_sales
  Regional quarterly summary:
  [table with region, quarter, sales_count, total_revenue, avg_revenue]

In [ ]:
logger.info("TASK 3: Load")
print("\n" + "="*70)
print("TASK 3: LOAD TO DATABASE")
print("="*70)

══════════════════════════════════════════════════════════════
 STEP 4 — ANALYTICAL QUERIES (demonstrated for you)

══════════════════════════════════════════════════════════════
These run after Task 3. They will only work if conn is defined.

EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
print("\n" + "="*70)
print("STEP 4: ANALYTICAL QUERIES")
print("="*70)

try:
    # Query 1: Top 5 products by revenue
    df_products = pd.read_sql("""
        SELECT product,
               SUM(quantity)  AS units_sold,
               ROUND(SUM(revenue), 2) AS total_revenue
        FROM   fact_sales
        GROUP BY product
        ORDER BY total_revenue DESC
        LIMIT 5
    """, conn)
    print("\nTop 5 Products by Revenue:")
    print(df_products.to_string(index=False))

    # Query 2: Rep performance
    df_reps = pd.read_sql("""
        SELECT rep_name,
               region,
               COUNT(*)               AS deals,
               ROUND(SUM(revenue), 2) AS total_revenue,
               ROUND(AVG(revenue), 2) AS avg_deal
        FROM   fact_sales
        GROUP BY rep_name, region
        ORDER BY total_revenue DESC
        LIMIT 8
    """, conn)
    print("\nTop Rep Performance:")
    print(df_reps.to_string(index=False))

    # Query 3: Deal size distribution
    df_deals = pd.read_sql("""
        SELECT deal_size,
               COUNT(*) AS count,
               ROUND(SUM(revenue), 2) AS total_revenue
        FROM   fact_sales
        GROUP BY deal_size
        ORDER BY total_revenue DESC
    """, conn)
    print("\nDeal Size Distribution:")
    print(df_deals.to_string(index=False))

except Exception as e:
    print(f"Query error: {e}")
    print("Make sure Task 3 is complete and 'conn' is defined.")

══════════════════════════════════════════════════════════════
 TASK 4 — QUALITY CHECK AND PIPELINE REPORT

══════════════════════════════════════════════════════════════
Instructions (two parts):

PART A -- Quality Checks:
Using df_clean, check:
  1. No null values in required cols: sale_id, rep_name, product, revenue
  2. All revenue values > 0
  3. sale_id is unique
  4. All regions in {"West", "East", "Central"}
Print each check as [PASS] or [FAIL] with the failure count.

PART B -- Pipeline Run Report:
Create a dict called pipeline_report with these keys:
  "run_timestamp"        : datetime.now().isoformat()
  "source_files"         : list(source_files.keys())
  "rows_extracted"       : len(df_raw)
  "rows_after_transform" : len(df_clean)
  "rows_loaded"          : len(df_clean)
  "quality_checks_passed": number of checks that passed
  "quality_checks_total" : 4
  "total_revenue"        : df_clean["revenue"].sum().round(2)
  "regions_covered"      : df_clean["region"].nunique()
  "date_range"           : str(df_clean["sale_date"].min().date()) + " to " + str(df_clean["sale_date"].max().date())
  "pipeline_status"      : "SUCCESS" if all quality checks pass, else "WARNING"

Print the report at the end.

In [ ]:
logger.info("TASK 4: Quality Check and Report")
print("\n" + "="*70)
print("TASK 4: QUALITY CHECK AND PIPELINE REPORT")
print("="*70)

══════════════════════════════════════════════════════════════
 FINAL OUTPUT — PRINT REPORT SUMMARY

══════════════════════════════════════════════════════════════

In [ ]:
print("\n" + "="*70)
print("PIPELINE RUN COMPLETE")
print("="*70)

try:
    for key, value in pipeline_report.items():
        print(f"  {key:<30}: {value}")
except NameError:
    print("Note: pipeline_report not yet defined. Complete Task 4 first.")

# Save report to JSON
import json
try:
    report_path = os.path.join(this_dir, "pipeline_run_report.json")
    # Convert any non-serializable types
    serializable_report = {}
    for k, v in pipeline_report.items():
        if isinstance(v, (pd.Timestamp, datetime)):
            serializable_report[k] = str(v)
        elif hasattr(v, "item"):   # numpy scalar
            serializable_report[k] = v.item()
        else:
            serializable_report[k] = v

    with open(report_path, "w") as f:
        json.dump(serializable_report, f, indent=2)
    logger.info(f"Report saved: {report_path}")
    print(f"\nReport saved to: {report_path}")
except NameError:
    print("Report not saved -- pipeline_report not defined yet.")
except Exception as e:
    print(f"Could not save report: {e}")

# Close connection if open
try:
    conn.close()
    logger.info("Database connection closed.")
except Exception:
    pass

print("\n" + "="*70)
print("PROJECT COMPLETE -- You have built a production-grade ETL pipeline!")
print("="*70)

══════════════════════════════════════════════════════════════
 CONCEPT 5 — HEALTH CHECK ENDPOINTS: THE PRODUCTION HEARTBEAT
══════════════════════════════════════════════════════════════
 ROYAL ROAD STANDARD — W8
 Every production service must expose two endpoints:

   GET /health  → "is the service process alive?"
                  Returns 200 {"status": "ok"} if the server is running.
                  Load balancers poll this every 5-30 seconds.

   GET /ready   → "is the service ready to accept traffic?"
                  Checks dependencies: database connected, models loaded, etc.
                  Returns 200 if all checks pass, 503 if any fail.

 Why this matters:
   - Kubernetes uses /health to decide whether to restart a pod
   - Load balancers use /ready to route traffic only to healthy instances
   - On-call engineers check these first when an alert fires

 Pattern: /health is always fast and never calls external systems.
          /ready can be slow — it actually tests dependencies.

 EXAMPLE ──────────────────────────────────────────────────────

In [ ]:
# Install: pip install fastapi uvicorn
# In production: uvicorn app:app --host 0.0.0.0 --port 8000
# We simulate the request/response cycle without running a server.

import sqlite3
import time
from datetime import datetime

# ── Simulated FastAPI app structure ──────────────────────────

def health_handler() -> dict:
    """GET /health — always returns 200 if process is alive."""
    return {"status": "ok", "timestamp": datetime.now().isoformat()}


def ready_handler(db_path: str = ":memory:") -> tuple[dict, int]:
    """GET /ready — checks all dependencies, returns 200 or 503."""
    checks = {}

    # Check 1: database connectivity
    try:
        conn = sqlite3.connect(db_path, timeout=2)
        conn.execute("SELECT 1")
        conn.close()
        checks["database"] = "ok"
    except Exception as e:
        checks["database"] = f"failed: {e}"

    # Check 2: memory (simplified simulation)
    checks["memory"] = "ok"

    all_ok = all(v == "ok" for v in checks.values())
    status_code = 200 if all_ok else 503
    return {"status": "ready" if all_ok else "not ready", "checks": checks}, status_code


def metrics_handler(request_counts: dict) -> dict:
    """GET /metrics — returns operational metrics."""
    return {
        "uptime_seconds": int(time.time() % 86400),
        "requests": request_counts,
        "timestamp": datetime.now().isoformat(),
    }


# Simulate endpoint calls
print("GET /health →", health_handler())

body, code = ready_handler()
print(f"GET /ready  → {code}", body)

counts = {"total": 1024, "success": 1018, "errors": 6}
print("GET /metrics →", metrics_handler(counts))

══════════════════════════════════════════════════════════════
 TASK 5 — ADD VERSION AND CIRCUIT BREAKER ENDPOINTS
══════════════════════════════════════════════════════════════
 Using the endpoint pattern above, implement two more handlers:

   version_handler(app_name, version, build_date) -> dict
     Returns: {"app": app_name, "version": version, "build_date": build_date}

   status_handler(db_path, error_rate_pct) -> tuple[dict, int]
     Checks the database (like ready_handler does).
     Also checks error_rate: if error_rate_pct > 5.0, add a warning.
     Returns 200 if db is ok, 503 if db fails.
     If db ok but error_rate > 5.0:
       {"status": "degraded", "warning": "error rate X.X% above threshold", "checks": {...}}

 Test with the starting data below and print all responses.

 Expected output:
     GET /version → {'app': 'etl-service', 'version': '2.1.4', 'build_date': '2025-01-15'}
     GET /status (healthy)   → 200 {'status': 'ok', 'checks': {'database': 'ok'}}
     GET /status (degraded)  → 200 {'status': 'degraded', 'warning': 'error rate 8.2% above threshold', ...}
     GET /status (db failed) → 503 {'status': 'not ready', 'checks': {'database': 'failed: ...'}}

 --- starting data ---

In [ ]:
import sqlite3, time
from datetime import datetime

app_name   = "etl-service"
version    = "2.1.4"
build_date = "2025-01-15"

db_ok      = ":memory:"          # valid db path
db_broken  = "/nonexistent/db"   # will fail
error_rate_normal   = 1.2        # below threshold
error_rate_high     = 8.2        # above threshold